# 🧠 LeetCode 853: Car Fleet

**Pattern:** Monotonic Stack (Sorting + Math)

- **Created:** 2026-02-28
- **Focus:** Calculating collisions by comparing arrival times
- **Tags:** `array` | `math` | `stack` | `sorting` | `medium` | `citi-prep`

## 📖 Problem Statement

There are `n` cars going to the same destination along a one-lane road. The destination is `target` miles away.

You are given two integer arrays `position` and `speed`, both of length `n`, where `position[i]` is the position of the $i^{th}$ car and `speed[i]` is the speed of the $i^{th}$ car (in miles per hour).

A car can never pass another car ahead of it, but it can catch up to it and drive bumper to bumper at the same speed. The faster car will slow down to match the slower car's speed. The distance between these two cars is ignored (they are assumed to have the same position). A car fleet is some non-empty set of cars driving at the same position and same speed.

Return the number of car fleets that will arrive at the destination.

### Example 1:
- **Input:** `target = 12`, `position = [10,8,0,5,3]`, `speed = [2,4,1,1,3]`
- **Output:** `3`
- **Explanation:**
  - The cars starting at 10 and 8 become a fleet, meeting each other at 12.
  - The car starting at 5 doesn't catch up to any other car, so it is a fleet by itself.
  - The cars starting at 0 and 3 become a fleet, meeting each other at 6. The fleet moves at speed 1 until it reaches target.

## 🧠 Mental Model & Intuition

Think of this purely as a math problem involving time.

If you are driving far behind me, but you calculate you will reach the destination at `3:00 PM`, and I calculate I will reach it at `4:00 PM`... logically, you MUST crash into me at some point before the destination, because it's a one-lane road and you are faster.

Once you crash into me, you are forced to slow down to my speed. You become part of my "fleet", and we both arrive together at `4:00 PM`.

**The Engine:**
1. Sort the cars by starting position so we look at the one closest to the target first.
2. Calculate exactly how long (Time = $\frac{Distance}{Speed}$) it will take each car to arrive if there were no traffic.
3. Process the cars backwards (closest to target down to furthest).
4. If a car's arrival time is $\le$ the arrival time of the fleet immediately ahead of it in the stack, they crash into each other and merge. If its arrival time is $>$, it's too slow to catch up, so it forms a brand new fleet.

## 🐢 Brute Force Approach

Simulate the movement of every car step-by-step (e.g., adding `speed` to `position` in a massive `while loop`). If two cars occupy the same position, merge them and update the faster car's speed to the slower car's speed. Keep simulating until all cars cross the target.

**Time:** $O(target \times N)$ or worse (Total simulation timeout) | **Space:** $O(N)$

In [ ]:
# Optimal Sorting + Stack Approach
def carFleet(target: int, position: list[int], speed: list[int]) -> int:
    # Zip them together so we don't lose the pairing, then sort by position descending
    # Reversing the sort means we process the car CLOSEST to the target first.
    pair = [[p, s] for p, s in zip(position, speed)]
    
    stack = []
    for p, s in sorted(pair)[::-1]:
        # Math: Time = Distance / Speed
        arrival_time = (target - p) / s
        
        stack.append(arrival_time)
        
        # If this car arrives FASTER or at the EXACT SAME TIME as the car ahead of it,
        # it will inevitably crash into the car ahead of it and join its fleet.
        if len(stack) >= 2 and stack[-1] <= stack[-2]:
            # It joins the fleet ahead, so we pop its independent arrival time.
            # (It adopts the slower arrival time of the car ahead, which is already at stack[-2])
            stack.pop()
            
    # The stack length represents the number of distinct fleets that never caught each other.
    return len(stack)

print("Car fleets for target=12: ", carFleet(12, [10,8,0,5,3], [2,4,1,1,3]))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N \log N)$. The dominant operation is sorting the initial array of pairs. The subsequent array iteration and stack population is $O(N)$ linear.
- **Space Complexity:** $O(N)$. We create a secondary array `pair` binding the tuples together, and use a `stack` array to track the bottleneck times.

## 🎤 Interview Q&A

### Q1: Why must we sort the array first, instead of just iterating left to right?
**Answer:** Because it's a one-lane road. Car A (at position 0) cannot possibly hit Car C (at position 10) unless it also successfully passes through Car B (at position 5). By sorting descending, the car closest to the target acts as the ultimate "bottleneck" for everyone behind it. Working backward guarantees we evaluate collisions accurately cascaded from the front of the line.

---

### Q2: Is the stack actually necessary? Could we just use a variable to track the slowest time?
**Answer:** Yes! The stack isn't strictly necessary here because we only ever compare against the single fleet immediately in front of us (`stack[-2]`). We could optimize space to $O(1)$ (ignoring the sort) by keeping a `max_time_blocker` variable. If `car_time > max_time_blocker`, we increment `fleet_count` and update `max_time_blocker = car_time`.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Physics Formula** | $T = D/S$ | `time = (target - p) / s` |
| **Bottleneck Propagation** | Processing from the most constrained state backwards. | Looping `[::-1]` |
| **Zipping** | Combining two parallel arrays into an array of Tuples so sorts don't break mappings. | `zip(pos, speed)` |

## 💼 The Citi Narrative

**Context:** Network Packet Batching & Congestion Control.

**Scenario:** Market data packets from different exchanges were arriving at our ingestion layer through a TCP socket. Certain fast, small packets emitted by the CME exchange would inevitably queued up right behind large "heavy" packets from ICE. Because the downstream ingestion layer processed them sequentially in a single thread, the fast packets functionally "crashed" into the slow packets and were processed as a single delayed batch (fleet).

**Impact:** Profiling "How many distinct processing batches will our microservice generate given this traffic burst pattern?" was an integration challenge. Implementing this `target - position / speed` array processing algorithm allowed us to simulate the exact batch-fleet congestion patterns without having to build a massive, complex time-step simulator, saving weeks of QA testing.

In [ ]:
# EXERCISE: Optimize to O(1) Space (Excluding Sort)
def carFleetOptimized(target: int, position: list[int], speed: list[int]) -> int:
    pair = sorted(zip(position, speed), reverse=True)
    fleets = 0
    slowest_time_ahead = 0
    
    for p, s in pair:
        time = (target - p) / s
        if time > slowest_time_ahead:
            fleets += 1
            slowest_time_ahead = time  # This car is too slow to catch up, forms new bottleneck
            
    return fleets

print("O(1) Memory variant fleets: ", carFleetOptimized(12, [10,8,0,5,3], [2,4,1,1,3]))

## 🎯 Summary: Key Takeaways

### The Pattern
**Monotonic Stack (Sorting + Math)** — Traversing ordered data dynamically resolving collisions based on theoretical arrival limits.

### When to Use It
- ✅ Time-distance collision detections (Batch congestion).
- ✅ Evaluating cascading system bottlenecks in pipeline processing.
- ❌ **Don't use when:** Two-dimensional vector intersection paths (Requires strict Euclidean math).

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(target \times N)$ | $O(N)$ |
| Optimal | $O(N \log N)$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Monotonic Stack (Sorting + Math)** and you've mastered one of the most common patterns in data engineering interviews. 🚀